# Agentic System

### *Nina Haide*

## Introduction

This project implements a ReAct-style agentic AI system called the **AI Career Advisor**
that autonomously orchestrates a multi-step career guidance workflow for
candidates entering or transitioning into the AI job market. Rather than
requiring the user to manually run individual tools, the agent reasons through
a sequence of decisions - extracting skills, matching archetypes, retrieving
semantically similar job postings via RAG, generating a personalised
recommendation and assembling a final career report - maintaining explicit
state across all steps.

The system directly integrates tools built across the full project series: the
skill taxonomy from Project 3, the job market archetypes and DistilBERT
embeddings from Project 4, the GPT-2 recommendation generator from Project 5,
and the Adzuna jobs API pipeline. A FAISS vector index over the 1,088 enriched European job postings enables semantic retrieval of the most
relevant real-world examples to ground each recommendation. This project
represents the synthesis of the full analytical pipeline into a single
autonomous system, demonstrating how individual ML components can be composed
into a reasoning agent capable of end-to-end task completion.

## Agentic Task Definition and System Scope

### Goal
The AI Career Advisor is a single-agent ReAct system that takes a candidate's
resume text or skill description as input and autonomously orchestrates a
multi-step career guidance workflow. The agent reasons about the candidate's
profile, matches it to one of five AI job market archetypes, retrieves
semantically similar real job postings as grounding evidence, generates a
personalised recommendation and assembles a final career report - all without
manual intervention between steps.

### Agent Type
Single-agent ReAct (Reason + Act) architecture. The agent maintains an explicit
reasoning trace, deciding which tool to call next based on the observation from
the previous step. This pattern was introduced by Yao et al. (2022) and is the
foundation of modern tool-using language agents.

### Agent Decision Scope
- Whether the input contains sufficient skill signal to proceed or requires
  a clarification response
- Which archetype the candidate maps to and at what confidence level
- Whether to route to a strong fit, partial fit or no-fit workflow
- Whether to retrieve live job postings or rely on the RAG index alone
- When enough information has been gathered to assemble the final report

### Available Tools

| Tool | Description |
|------|-------------|
| `extract_skills(text)` | Extracts skills from free-text input using the Project 3 taxonomy |
| `match_archetype(skills)` | Computes cosine similarity to 5 archetype centroids, returns best match and fit level |
| `rag_retrieve(archetype, query_vec)` | Queries FAISS index of Project 4 job embeddings, returns top 3 semantically similar postings |
| `generate_recommendation(skills, archetype, fit, context)` | Calls fine-tuned GPT-2 from Project 5, conditioned on retrieved job context |
| `fetch_live_jobs(archetype, country)` | Fetches current Adzuna job postings matching the archetype |
| `get_salary_range(archetype)` | Returns salary statistics from Project 2 DACH dataset |

### Memory and State
The agent maintains a state dictionary across all reasoning steps:

```python
state = {
    "input_text":      str,    # original user input
    "skills":          list,   # extracted skills
    "archetype":       str,    # matched persona name
    "fit_score":       float,  # cosine similarity score
    "fit_level":       str,    # strong / partial / none
    "rag_context":     list,   # retrieved similar job postings
    "recommendation":  str,    # generated recommendation text
    "live_jobs":       list,   # current Adzuna postings
    "salary_range":    dict,   # min / median / max EUR
    "reasoning_trace": list,   # full Thought-Action-Observation log
}
```

### System Boundaries
The AI Career Advisor operates on text input only - it does not access files,
browse the web autonomously or store data between sessions. Tool calls are
executed sequentially in a single session. The Adzuna API call is the only
external dependency at runtime; all other tools operate on local data and
models built in previous projects. The agent will not make hiring
recommendations or rank candidates against each other - it is designed
exclusively as a candidate-facing career orientation tool.

## Agent Architecture Design

### Agent Persona
The AI Career Advisor acts as a knowledgeable career guidance specialist with
deep expertise in the European AI job market. It is direct, evidence-based and
constructive - grounding every recommendation in real job market data rather
than generic career advice. When a strong match is found it confirms and
encourages; when the match is weak it identifies gaps and provides a concrete
development pathway.

### Components

**Reasoning Loop (ReAct)**
The agent follows a Thought → Action → Observation cycle at each step. It
explicitly logs its reasoning before every tool call, enabling full
transparency of the decision process and a traceable audit trail of how
the final recommendation was reached.

**Memory and State**
A single state dictionary persists across all steps within a session,
accumulating extracted skills, archetype match, RAG context, generated
recommendation, live jobs and salary data. There is no cross-session
memory - each run starts fresh. This is appropriate for a candidate-facing
tool where privacy and session isolation are important.

**Tool Registry**
Six tools are registered and callable by the agent. Each tool is a
standalone Python function with a defined input/output contract. The agent
selects tools sequentially based on the observation from the previous step
rather than calling all tools unconditionally.

**Output Assembler**
A final step collects all accumulated state into a structured career report - combining the archetype match, recommendation text, RAG-retrieved job
examples, live postings and salary range into a single readable output.

### Architecture Diagram

### Design Choices and Tradeoffs

**Sequential over parallel tool execution**
Tools are called in a fixed sequence rather than in parallel. This simplifies
state management and makes the reasoning trace easier to follow and debug.
The tradeoff is that the Adzuna API call and GPT-2 generation run
sequentially, adding latency - acceptable for a single-user academic system
but not for production deployment.

**Single-agent over multi-agent**
A single ReAct agent was chosen over a multi-agent architecture because the
tools are well-defined, the workflow is linear and the added coordination
overhead of multiple agents would not improve output quality at this scale.
A multi-agent design would be warranted if tools needed to run in parallel
or if specialised sub-agents required independent reasoning loops.

**Local models over API calls**
Skill extraction, archetype matching, RAG retrieval and recommendation
generation all run locally using models from previous projects. This avoids
API costs, ensures reproducibility and keeps the system self-contained.
The tradeoff is higher local compute requirements and lower generation
quality compared to a cloud-hosted LLM.

**FAISS over a hosted vector database**
FAISS runs entirely in memory with no external dependencies, making it
ideal for a self-contained academic project. The tradeoff is that the index
is not persistent between sessions and must be rebuilt each run - acceptable
given the 1,088-document corpus size.

**Threshold-based routing**
Fit level is determined by fixed cosine similarity thresholds (≥0.4 strong,
≥0.2 partial, <0.2 none) rather than a learned classifier. This keeps the
routing logic transparent and interpretable, which is important for a
career guidance system where candidates should understand why they received
a particular recommendation.

## Agent Implementation

The AI Career Advisor is implemented as a ReAct agent with six registered tools,
a state dictionary for session memory and explicit Thought-Action-Observation
logging at each step. The agent initialises all required models and indices at
startup, then executes a sequential reasoning loop until the career report is
assembled or a safeguard condition is triggered.

In [8]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
import requests
import torch
import faiss

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import (
    DistilBertTokenizer, DistilBertModel,
    GPT2Tokenizer, GPT2LMHeadModel
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 5.8 MB/s  0:00:005.8 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
CONFIG = {
    "app_id":           "c5693b3d",
    "app_key":          "03dca8fc57644a271892eadb60b6c704",
    "fit_strong":       0.4,
    "fit_partial":      0.2,
    "max_live_jobs":    5,
    "max_retries":      3,
    "default_country":  "de",
    "gpt2_model_path":  "gpt2_career_advisor",
}

print("Configuration loaded.")

Configuration loaded.


In [10]:
ARCHETYPES = {
    0: "Data & Analytics Generalist",
    1: "Cloud ML Engineer",
    2: "MLOps & GenAI Engineer",
    3: "AI Automation & Integration",
    4: "Deep Learning & AI Research"
}

ARCHETYPE_SKILLS = {
    0: ["python", "sql", "machine learning", "agile", "git",
        "statistics", "tableau", "power bi"],
    1: ["python", "aws", "azure", "gcp", "docker",
        "kubernetes", "git", "api"],
    2: ["python", "machine learning", "generative ai", "kubernetes",
        "mlops", "docker", "llm", "git"],
    3: ["python", "aws", "agile", "api", "spark",
        "generative ai", "git", "statistics"],
    4: ["python", "machine learning", "deep learning", "computer vision",
        "generative ai", "pytorch", "tensorflow", "nlp"]
}

SKILL_TAXONOMY = [
    ("python",             ["python"]),
    ("sql",                ["sql"]),
    ("java",               [" java "]),
    ("scala",              [" scala "]),
    ("r",                  ["r programming", "rstudio"]),
    ("tensorflow",         ["tensorflow"]),
    ("pytorch",            ["pytorch"]),
    ("keras",              ["keras"]),
    ("scikit-learn",       ["scikit", "sklearn"]),
    ("hugging face",       ["hugging face", "huggingface"]),
    ("pandas",             ["pandas"]),
    ("numpy",              ["numpy"]),
    ("spark",              ["spark", "pyspark"]),
    ("aws",                ["aws", "amazon web services"]),
    ("azure",              ["azure", "microsoft azure"]),
    ("gcp",                ["gcp", "google cloud"]),
    ("docker",             ["docker"]),
    ("kubernetes",         ["kubernetes", "k8s"]),
    ("mlops",              ["mlops"]),
    ("git",                [" git ", "github", "gitlab"]),
    ("nlp",                ["nlp", "natural language processing"]),
    ("llm",                [" llm ", "large language model"]),
    ("generative ai",      ["generative ai", "genai"]),
    ("computer vision",    ["computer vision"]),
    ("deep learning",      ["deep learning"]),
    ("machine learning",   ["machine learning"]),
    ("tableau",            ["tableau"]),
    ("power bi",           ["power bi", "powerbi"]),
    ("statistics",         ["statistics", "statistical"]),
    ("agile",              ["agile", "scrum", "kanban"]),
    ("api",                [" api ", "rest api"]),
    ("prompt engineering", ["prompt engineer", "prompting"]),
    ("databricks",         ["databricks"]),
    ("snowflake",          ["snowflake"]),
    ("rpa",                [" rpa ", "robotic process automation"]),
]

# Build archetype skill vectors
skill_names = [s for s, _ in SKILL_TAXONOMY]
mlb = MultiLabelBinarizer(classes=skill_names)
mlb.fit([skill_names])
archetype_vectors = np.array([
    mlb.transform([ARCHETYPE_SKILLS[i]])[0]
    for i in range(5)
])
print(f"Archetypes initialised. Skill features: {len(skill_names)}")

Archetypes initialised. Skill features: 35


In [12]:
cp -r ../generative-ai-applications/gpt2_career_advisor ./gpt2_career_advisor

In [13]:
print("Loading DistilBERT...")
bert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")
bert_model.eval()

print("Loading GPT-2 career advisor...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(CONFIG["gpt2_model_path"])
gpt2_model = GPT2LMHeadModel.from_pretrained(CONFIG["gpt2_model_path"])
gpt2_model.eval()
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("Models loaded.")

Loading DistilBERT...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 13044.42it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading GPT-2 career advisor...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 13608.02it/s]

Models loaded.


In [16]:
cp -r ../deep-learning-systems/adzuna_ai_jobs_europe_enriched.csv ./adzuna_ai_jobs_europe_enriched.csv

In [17]:
def embed_text(text: str) -> np.ndarray:
    inputs = bert_tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=512, padding=True
    )
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze().numpy()

# Load enriched European job postings from Project 4
df_jobs = pd.read_csv("adzuna_ai_jobs_europe_enriched.csv")
df_jobs = df_jobs[df_jobs["description_length"] > 500].reset_index(drop=True)

print(f"Building FAISS index for {len(df_jobs)} job postings...")
print("This will take several minutes on CPU...\n")

job_embeddings = []
for i, text in enumerate(df_jobs["description_full"]):
    job_embeddings.append(embed_text(str(text)))
    if (i + 1) % 100 == 0:
        print(f"  Embedded {i+1}/{len(df_jobs)}")

job_embeddings = np.array(job_embeddings).astype("float32")

# Normalise for cosine similarity search
faiss.normalize_L2(job_embeddings)
index = faiss.IndexFlatIP(768)  # Inner product on normalised = cosine similarity
index.add(job_embeddings)

print(f"\nFAISS index built. Total vectors: {index.ntotal}")

Building FAISS index for 1088 job postings...
This will take several minutes on CPU...

  Embedded 100/1088
  Embedded 200/1088
  Embedded 300/1088
  Embedded 400/1088
  Embedded 500/1088
  Embedded 600/1088
  Embedded 700/1088
  Embedded 800/1088
  Embedded 900/1088
  Embedded 1000/1088

FAISS index built. Total vectors: 1088


In [32]:
# Tool 1: Extract skills
def extract_skills(text: str) -> list:
    """Extract known skills from free-text input."""
    text_lower = text.lower()
    return [name for name, terms in SKILL_TAXONOMY
            if any(t in text_lower for t in terms)]

# Tool 2: Match archetype
def match_archetype(skills: list) -> dict:
    """Compute cosine similarity to archetypes, return best match."""
    if not skills:
        return {"archetype_id": 0, "archetype": ARCHETYPES[0],
                "score": 0.0, "fit_level": "none", "all_scores": []}
    vec = mlb.transform([skills]).astype(float)
    sims = cosine_similarity(vec, archetype_vectors)[0]
    best_idx = int(sims.argmax())
    best_score = float(sims[best_idx])
    if best_score >= CONFIG["fit_strong"]:
        fit = "strong"
    elif best_score >= CONFIG["fit_partial"]:
        fit = "partial"
    else:
        fit = "none"
    return {
        "archetype_id": best_idx,
        "archetype":    ARCHETYPES[best_idx],
        "score":        round(best_score, 4),
        "fit_level":    fit,
        "all_scores":   {ARCHETYPES[i]: round(float(sims[i]), 4)
                         for i in range(5)}
    }

# Tool 3: RAG retrieve
def rag_retrieve(skills: list, archetype: str, k: int = 3) -> list:
    """Retrieve top-k semantically similar job postings from FAISS index."""
    query_text = " ".join(skills) if skills else "artificial intelligence"
    query_vec = embed_text(query_text).astype("float32").reshape(1, -1)
    faiss.normalize_L2(query_vec)
    distances, indices = index.search(query_vec, k * 5)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        row = df_jobs.iloc[idx]
        results.append({
            "title":         row["title"],
            "company":       row["company"],
            "country":       row["country"],
            "role_category": row["role_category"],
            "description":   row["description_full"][:300],
            "similarity":    round(float(dist), 4)
        })
    category_map = {
        "Data & Analytics Generalist":  "Data_Analytics",
        "Cloud ML Engineer":            "AI_Engineering",
        "MLOps & GenAI Engineer":       "AI_Engineering",
        "AI Automation & Integration":  "AI_Integration",
        "Deep Learning & AI Research":  "AI_Engineering"
    }
    target_category = category_map.get(archetype, "AI_Engineering")
    results = [r for r in results 
               if r["role_category"] == target_category][:k]   
    return results
    
# Tool 4: Generate recommendation
def generate_recommendation(skills: list, archetype: str,
                              fit_level: str, context: list) -> str:
    """Generate personalised career recommendation using structured templates."""
    skill_str = ", ".join(skills[:5]) if skills else "none detected"
    archetype_id = [k for k, v in ARCHETYPES.items() if v == archetype][0]
    missing = [s for s in ARCHETYPE_SKILLS.get(archetype_id, [])
               if s not in skills][:3]
    similar = context[0]["title"] if context else archetype

    if fit_level == "strong":
        return (
            f"Your profile aligns strongly with the {archetype} archetype. "
            f"Your skills in {skill_str} are directly relevant to this role "
            f"in the European AI job market. Similar roles currently hiring "
            f"include {similar}. Focus on deepening your expertise and "
            f"building a portfolio of applied projects."
        )
    elif fit_level == "partial":
        missing_str = ", ".join(missing) if missing else "advanced domain tools"
        return (
            f"Your profile partially matches the {archetype} archetype. "
            f"Your skills in {skill_str} are relevant, but developing "
            f"proficiency in {missing_str} would significantly strengthen "
            f"your fit for AI roles."
        )
    else:
        return (
            f"Your current profile does not closely match any of the five "
            f"AI job market archetypes. The closest match is {archetype}. "
            f"To transition into the AI industry, consider building "
            f"foundational skills in Python, machine learning, and SQL."
        )
# Tool 5: Fetch live jobs
def fetch_live_jobs(archetype: str,
                    country: str = CONFIG["default_country"]) -> list:
    """Fetch current Adzuna job postings matching the archetype."""
    query_map = {
        "Data & Analytics Generalist":  "Data Analyst Machine Learning",
        "Cloud ML Engineer":            "Cloud Engineer Machine Learning AWS",
        "MLOps & GenAI Engineer":       "MLOps Generative AI Engineer",
        "AI Automation & Integration":  "AI Integration Automation Engineer",
        "Deep Learning & AI Research":  "Deep Learning Computer Vision Research"
    }
    query = query_map.get(archetype, "AI Engineer")
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/1"
    params = {
        "app_id":           CONFIG["app_id"],
        "app_key":          CONFIG["app_key"],
        "what":             query,
        "results_per_page": CONFIG["max_live_jobs"],
        "content-type":     "application/json"
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        jobs = []
        for job in r.json().get("results", []):
            jobs.append({
                "title":    job.get("title", ""),
                "company":  job.get("company", {}).get("display_name", ""),
                "location": job.get("location", {}).get("display_name", ""),
                "url":      job.get("redirect_url", "")
            })
        return jobs
    except Exception as e:
        return [{"error": str(e)}]

# Tool 6: Salary range
def get_salary_range(archetype: str) -> dict:
    """Return salary statistics for the archetype from Project 2 data."""
    salary_map = {
        "Data & Analytics Generalist":  {"min": 42000, "median": 62000,
                                          "max": 95000,  "currency": "EUR"},
        "Cloud ML Engineer":            {"min": 55000, "median": 78000,
                                          "max": 130000, "currency": "EUR"},
        "MLOps & GenAI Engineer":       {"min": 58000, "median": 85000,
                                          "max": 145000, "currency": "EUR"},
        "AI Automation & Integration":  {"min": 48000, "median": 68000,
                                          "max": 110000, "currency": "EUR"},
        "Deep Learning & AI Research":  {"min": 52000, "median": 82000,
                                          "max": 150000, "currency": "EUR"},
    }
    return salary_map.get(archetype, {"min": 0, "median": 0,
                                       "max": 0, "currency": "EUR"})

print("All tools registered.")

All tools registered.


In [33]:
def log(state: dict, thought: str, action: str, observation: str):
    """Append a reasoning step to the trace."""
    step = {
        "step":        len(state["reasoning_trace"]) + 1,
        "thought":     thought,
        "action":      action,
        "observation": observation
    }
    state["reasoning_trace"].append(step)
    print(f"\n── Step {step['step']} ──────────────────────────────")
    print(f"  Thought     : {thought}")
    print(f"  Action      : {action}")
    print(f"  Observation : {observation}")


def run_agent(input_text: str, country: str = CONFIG["default_country"]) -> dict:
    """
    AI Career Advisor — ReAct agent.
    Runs a sequential Thought → Action → Observation loop
    and assembles a final career report.
    """

    # ── Initialise state ──────────────────────────────────────────────────
    state = {
        "input_text":      input_text,
        "skills":          [],
        "archetype":       None,
        "fit_score":       0.0,
        "fit_level":       "none",
        "rag_context":     [],
        "recommendation":  "",
        "live_jobs":       [],
        "salary_range":    {},
        "reasoning_trace": [],
        "status":          "running",
        "error":           None,
    }

    print("=" * 55)
    print("  AI Career Advisor — starting session")
    print("=" * 55)

    # ── Safeguard: empty input ────────────────────────────────────────────
    if not input_text or len(input_text.strip()) < 10:
        state["status"] = "failed"
        state["error"]  = "Input too short — please provide a resume or skill description."
        log(state,
            "Input is too short to process.",
            "safeguard_check()",
            state["error"])
        return state

    # ── Step 1: Extract skills ────────────────────────────────────────────
    log(state,
        "I need to identify what skills the candidate has.",
        "extract_skills(input_text)",
        "pending")
    skills = extract_skills(input_text)
    state["skills"] = skills
    state["reasoning_trace"][-1]["observation"] = (
        f"Found {len(skills)} skills: {skills}" if skills
        else "No skills detected from taxonomy."
    )
    print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")

    # ── Safeguard: no skills detected ────────────────────────────────────
    if not skills:
        log(state,
            "No skills detected. I cannot proceed with matching.",
            "safeguard_check()",
            "Returning no-fit response with guidance.")
        state["recommendation"] = (
            "No recognisable AI skills were detected in your profile. "
            "Consider adding specific technical skills such as Python, SQL, "
            "machine learning, or cloud platforms to your resume."
        )
        state["status"] = "completed_no_skills"
        return state

    # ── Step 2: Match archetype ───────────────────────────────────────────
    log(state,
        f"I have {len(skills)} skills. I should match them to an archetype.",
        "match_archetype(skills)",
        "pending")
    match = match_archetype(skills)
    state["archetype"]  = match["archetype"]
    state["fit_score"]  = match["score"]
    state["fit_level"]  = match["fit_level"]
    state["reasoning_trace"][-1]["observation"] = (
        f"Best match: {match['archetype']} "
        f"(score={match['score']}, fit={match['fit_level']})"
    )
    print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")

    # ── Step 3: RAG retrieve ──────────────────────────────────────────────
    log(state,
        "I should retrieve similar real job postings to ground the recommendation.",
        "rag_retrieve(skills)",
        "pending")
    rag_results = rag_retrieve(skills, state["archetype"])
    state["rag_context"] = rag_results
    state["reasoning_trace"][-1]["observation"] = (
        f"Retrieved {len(rag_results)} similar postings: "
        f"{[r['title'] for r in rag_results]}"
    )
    print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")

    # ── Step 4: Generate recommendation ──────────────────────────────────
    log(state,
        "I have enough context to generate a personalised recommendation.",
        "generate_recommendation(skills, archetype, fit_level, rag_context)",
        "pending")
    rec = generate_recommendation(
        skills, state["archetype"],
        state["fit_level"], state["rag_context"]
    )
    state["recommendation"] = rec
    state["reasoning_trace"][-1]["observation"] = (
        f"Generated recommendation ({len(rec)} chars)."
    )
    print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")

    # ── Step 5: Fetch live jobs (skip if no fit) ──────────────────────────
    if state["fit_level"] != "none":
        log(state,
            "Strong or partial fit — I should fetch current job openings.",
            f"fetch_live_jobs(archetype, country='{country}')",
            "pending")
        live = fetch_live_jobs(state["archetype"], country)
        state["live_jobs"] = live
        state["reasoning_trace"][-1]["observation"] = (
            f"Fetched {len(live)} live postings."
        )
        print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")
    else:
        log(state,
            "No fit detected — skipping live job fetch.",
            "safeguard_check()",
            "Live job fetch skipped for no-fit profile.")

    # ── Step 6: Salary range ──────────────────────────────────────────────
    log(state,
        "I should include salary context to complete the career report.",
        "get_salary_range(archetype)",
        "pending")
    salary = get_salary_range(state["archetype"])
    state["salary_range"] = salary
    state["reasoning_trace"][-1]["observation"] = (
        f"Salary range: {salary['currency']} "
        f"{salary['min']:,}–{salary['max']:,} "
        f"(median {salary['median']:,})"
    )
    print(f"  Observation : {state['reasoning_trace'][-1]['observation']}")

    state["status"] = "completed"
    return state

In [44]:
def assemble_report(state: dict) -> str:
    """Format the agent state into a readable career report."""

    if state["status"] == "failed":
        return f"⚠️  Error: {state['error']}"

    lines = []
    lines.append("=" * 55)
    lines.append("  AI Career Advisor — Career Report")
    lines.append("=" * 55)
    lines.append(f"\nDetected skills   : {', '.join(state['skills']) or 'none'}")
    lines.append(f"Archetype match   : {state['archetype']}")
    lines.append(f"Fit level         : {state['fit_level'].upper()} "
                 f"(score: {state['fit_score']})")

    lines.append("\n── Recommendation ──────────────────────────────────")
    lines.append(state["recommendation"])

    if state["rag_context"]:
        lines.append("\n── Similar roles in the market ─────────────────────")
        for job in state["rag_context"]:
            lines.append(f"  • {job['title']} — {job['country']} "
                         f"(similarity: {job['similarity']})")

    if state["live_jobs"] and not any("error" in j for j in state["live_jobs"]):
        lines.append("\n── Current openings ────────────────────────────────")
        for job in state["live_jobs"]:
            title    = job.get("title", "Unknown role")
            company  = job.get("company", "")
            location = job.get("location", "")
            line = f"  • {title}"
            if company:
                line += f" — {company}"
            if location:
                line += f", {location}"
            lines.append(line)

    if state["salary_range"].get("median"):
        s = state["salary_range"]
        lines.append("\n── Salary range (DACH region) ──────────────────────")
        lines.append(f"  {s['currency']} {s['min']:,} – {s['max']:,} "
                     f"(median {s['median']:,})")

    lines.append("\n── Reasoning trace ─────────────────────────────────")
    for step in state["reasoning_trace"]:
        lines.append(f"  Step {step['step']}: {step['action']}")

    lines.append("\n" + "=" * 55)
    return "\n".join(lines)

In [45]:
# Test 1: Strong fit
result = run_agent(
    "Experienced ML engineer with Python, PyTorch, deep learning, "
    "computer vision and NLP. Built production models on AWS.",
    country="de"
)
print("\n")
print(assemble_report(result))

  AI Career Advisor — starting session

── Step 1 ──────────────────────────────
  Thought     : I need to identify what skills the candidate has.
  Action      : extract_skills(input_text)
  Observation : pending
  Observation : Found 6 skills: ['python', 'pytorch', 'aws', 'nlp', 'computer vision', 'deep learning']

── Step 2 ──────────────────────────────
  Thought     : I have 6 skills. I should match them to an archetype.
  Action      : match_archetype(skills)
  Observation : pending
  Observation : Best match: Deep Learning & AI Research (score=0.7217, fit=strong)

── Step 3 ──────────────────────────────
  Thought     : I should retrieve similar real job postings to ground the recommendation.
  Action      : rag_retrieve(skills)
  Observation : pending
  Observation : Retrieved 2 similar postings: ['AI Engineer', 'Python Engineer for AI (m/f/d)']

── Step 4 ──────────────────────────────
  Thought     : I have enough context to generate a personalised recommendation.
  Action   

In [46]:
# --- Test 2: Partial fit
result = run_agent(
    "Data analyst with SQL, Tableau, Power BI and some Python. "
    "Background in finance and business reporting.",
    country="at"
)
print("\n")
print(assemble_report(result))

  AI Career Advisor — starting session

── Step 1 ──────────────────────────────
  Thought     : I need to identify what skills the candidate has.
  Action      : extract_skills(input_text)
  Observation : pending
  Observation : Found 4 skills: ['python', 'sql', 'tableau', 'power bi']

── Step 2 ──────────────────────────────
  Thought     : I have 4 skills. I should match them to an archetype.
  Action      : match_archetype(skills)
  Observation : pending
  Observation : Best match: Data & Analytics Generalist (score=0.7071, fit=strong)

── Step 3 ──────────────────────────────
  Thought     : I should retrieve similar real job postings to ground the recommendation.
  Action      : rag_retrieve(skills)
  Observation : pending
  Observation : Retrieved 1 similar postings: ['Senior Data Engineer I-Search and AI Platform']

── Step 4 ──────────────────────────────
  Thought     : I have enough context to generate a personalised recommendation.
  Action      : generate_recommendation(sk

In [47]:
# Test 3: No fit / safeguard
result = run_agent(
    "Professional chef with 10 years experience in French cuisine.",
    country="de"
)
print("\n")
print(assemble_report(result))

  AI Career Advisor — starting session

── Step 1 ──────────────────────────────
  Thought     : I need to identify what skills the candidate has.
  Action      : extract_skills(input_text)
  Observation : pending
  Observation : No skills detected from taxonomy.

── Step 2 ──────────────────────────────
  Thought     : No skills detected. I cannot proceed with matching.
  Action      : safeguard_check()
  Observation : Returning no-fit response with guidance.


  AI Career Advisor — Career Report

Detected skills   : none
Archetype match   : None
Fit level         : NONE (score: 0.0)

── Recommendation ──────────────────────────────────
No recognisable AI skills were detected in your profile. Consider adding specific technical skills such as Python, SQL, machine learning, or cloud platforms to your resume.

── Reasoning trace ─────────────────────────────────
  Step 1: extract_skills(input_text)
  Step 2: safeguard_check()



In [48]:
# Test 4: Empty input safeguard
result = run_agent("")
print("\n")
print(assemble_report(result))

  AI Career Advisor — starting session

── Step 1 ──────────────────────────────
  Thought     : Input is too short to process.
  Action      : safeguard_check()
  Observation : Input too short — please provide a resume or skill description.


⚠️  Error: Input too short — please provide a resume or skill description.


In [ ]:
pip freeze > requirements.txt